# Module 1 — Full Fine-tune YOLO11-Seg trên MEDISEG (Google Colab)
Dựa trên mediseg_split_augment.py của bạn, tách thành pipeline 4 bước.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. CONFIG — sửa 3 đường dẫn này cho đúng Drive của bạn

- `CODE_DIR`: thư mục chứa code project (nơi có `training/` và `src/`) trên Drive.
- `DATA_ZIP`: đường dẫn tới file `mediseg_yolo.zip` bạn đã upload lên Drive.
- `EXPERIMENTS_DRIVE_DIR`: nơi lưu checkpoint/kết quả train trên Drive (để không mất khi session bị ngắt).

In [ ]:
CODE_DIR = "/content/drive/MyDrive/MEDISEG/module1_segmentation_yolov11_v2"  # <-- EDIT
DATA_ZIP = "/content/drive/MyDrive/MEDISEG/mediseg_yolo.zip"                 # <-- EDIT
EXPERIMENTS_DRIVE_DIR = "/content/drive/MyDrive/MEDISEG/experiments"        # <-- EDIT (sẽ tự tạo nếu chưa có)

DATA_DIR = "/content/mediseg_yolo"  # giải nén vào ổ local của Colab để I/O nhanh, không cần sửa

## 3. Giải nén data vào /content (ổ local, nhanh hơn Drive)

In [ ]:
import os
os.makedirs(DATA_DIR, exist_ok=True)
!unzip -q -o "{DATA_ZIP}" -d "{DATA_DIR}"
print("--- Nội dung DATA_DIR ---")
!ls "{DATA_DIR}"

# Một số file zip khi giải nén sẽ tạo thêm 1 lớp thư mục con trùng tên (vd DATA_DIR/mediseg_yolo/images/...).
# Nếu thấy vậy, tự động đưa nội dung ra ngoài 1 cấp cho khớp cấu trúc mong đợi.
inner = os.path.join(DATA_DIR, "mediseg_yolo")
if os.path.isdir(inner) and not os.path.isdir(os.path.join(DATA_DIR, "images")):
    print(f"[fix] Phát hiện thư mục lồng {inner}, đang đưa nội dung ra ngoài...")
    !mv "{inner}"/* "{DATA_DIR}"/
    !rmdir "{inner}"
    !ls "{DATA_DIR}"

## 4. Vào thư mục code + cài thư viện

In [ ]:
%cd {CODE_DIR}/training/segmentation_yolov11_full_finetune
!pip install -q -r requirements.txt

## 5. Vá lại config.py và data.yaml cho đúng đường dẫn Colab

Ô này tự động sửa `OUTPUT_DIR`, `EXPERIMENTS_ROOT` trong `config.py` và dòng `path:` trong `data.yaml` — **không cần** tự mở file sửa tay.

In [ ]:
import re
from pathlib import Path

config_path = Path(f"{CODE_DIR}/src/segmentation/config.py")
text = config_path.read_text(encoding="utf-8")

text = re.sub(
    r'OUTPUT_DIR = Path\(.*?\).*',
    f'OUTPUT_DIR = Path("{DATA_DIR}")',
    text,
    count=1,
)
text = re.sub(
    r'EXPERIMENTS_ROOT = Path\(.*?\).*',
    f'EXPERIMENTS_ROOT = Path("{EXPERIMENTS_DRIVE_DIR}")',
    text,
    count=1,
)
config_path.write_text(text, encoding="utf-8")
print("[ok] Đã sửa config.py:")
for line in text.splitlines():
    if line.strip().startswith(("OUTPUT_DIR", "EXPERIMENTS_ROOT")):
        print(" ", line)

yaml_path = Path(DATA_DIR) / "data.yaml"
yaml_text = yaml_path.read_text(encoding="utf-8")
yaml_text = re.sub(r'^path:.*$', f'path: {DATA_DIR}', yaml_text, count=1, flags=re.MULTILINE)
yaml_path.write_text(yaml_text, encoding="utf-8")
print("\n[ok] Đã sửa data.yaml:")
print(yaml_text)

## 6. Kiểm tra nhanh trước khi train (tuỳ chọn nhưng nên chạy)

In [ ]:
n_train = len(list((Path(DATA_DIR) / "images" / "train").glob("*.*")))
n_val = len(list((Path(DATA_DIR) / "images" / "val").glob("*.*")))
n_test = len(list((Path(DATA_DIR) / "images" / "test").glob("*.*")))
print(f"train: {n_train} | val: {n_val} | test: {n_test}")
assert n_train > 0 and n_val > 0, "Thiếu ảnh train/val — kiểm tra lại bước giải nén ở mục 3."

## 7. Train (full fine-tune YOLOv11-Seg)

In [ ]:
!python train/train.py

## 8. Evaluate trên tập test (mask mAP@0.5:0.95)

In [ ]:
!python evaluation/evaluate.py

## 9. (Tuỳ chọn) Kiểm tra lại checkpoint đã lưu vào Drive

In [ ]:
!ls -la "{EXPERIMENTS_DRIVE_DIR}"